# Cleaning for EDA

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer
from sklearn.compose import ColumnTransformer

## Overview

In [2]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "data_raw"

MODEL_DATA_DIR = DATA_DIR / "data_model"
MODEL_DATA_DIR.mkdir(exist_ok=True)

EDA_DATA_DIR = DATA_DIR / "data_eda"
EDA_DATA_DIR.mkdir(exist_ok=True)

In [3]:
df_raw = pd.read_csv(RAW_DATA_DIR / "train.csv")

In [4]:
df = df_raw.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [5]:
pd.set_option('display.max_columns', None)
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## First Treatments

### Treatment of String Values

In [6]:
columns_object = df.select_dtypes(include='object')
columns_str = columns_object.convert_dtypes(convert_string=True)

df[columns_object.columns] = columns_str

/tmp/ipykernel_11029/1109026682.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columns_object = df.select_dtypes(include='object')


In [7]:

df.columns = df.columns.str.upper()
columns_str = df.select_dtypes(include='string')

for col in columns_str.columns:
    df[col] = df[col].str.upper()

### Treatment of Null Values

In [8]:
null_values = df.isnull().mean()*100
null_values[null_values > 0].sort_values(ascending=False).round(2)

POOLQC          99.52
MISCFEATURE     96.30
ALLEY           93.77
FENCE           80.75
MASVNRTYPE      59.73
FIREPLACEQU     47.26
LOTFRONTAGE     17.74
GARAGETYPE       5.55
GARAGEYRBLT      5.55
GARAGEFINISH     5.55
GARAGEQUAL       5.55
GARAGECOND       5.55
BSMTEXPOSURE     2.60
BSMTFINTYPE2     2.60
BSMTQUAL         2.53
BSMTCOND         2.53
BSMTFINTYPE1     2.53
MASVNRAREA       0.55
ELECTRICAL       0.07
dtype: float64

In [9]:
high_missing_cols = null_values[null_values > 40]
medium_missing_cols = null_values[(null_values < 40) & (null_values > 10)]
low_missing_cols = null_values[null_values < 10]

df = df.drop(high_missing_cols.index, axis=1)
df["GARAGEYRBLT"] = df["GARAGEYRBLT"].fillna(0)

df = df.dropna(subset=low_missing_cols.index)

In [10]:
imputer = KNNImputer(n_neighbors=5, weights='distance')
df[medium_missing_cols.index] = imputer.fit_transform(df[medium_missing_cols.index])

## Classification of Variables

In [11]:
df['TOTALLIVINGSF'] = df['TOTALBSMTSF'] + df['1STFLRSF'] + df['2NDFLRSF']
df['HOUSE_AGE'] = df['YRSOLD'] - df['YEARBUILT']
df['REMODEL_AGE'] = df['YRSOLD'] - df['YEARREMODADD']
df['GARAGE_PROXY'] = df['GARAGECARS'] * df['GARAGEAREA']
df['BATHROOM_INDEX'] = df['FULLBATH'] + 0.5*df['HALFBATH'] + 0.5*df['BSMTFULLBATH']
df['HASBASEMENT'] = (df['TOTALBSMTSF'] > 0).astype(int)
df['HASGARAGE'] = (df['GARAGEAREA'] > 0).astype(int)
df['HASFIREPLACE'] = (df['FIREPLACES'] > 0).astype(int)
df['HASPOOL'] = (df['POOLAREA'] > 0).astype(int)
df['TOTALPORCHSF'] = df['OPENPORCHSF'] + df['SCREENPORCH'] + df['3SSNPORCH']


cols_to_drop = [
    'TOTALBSMTSF', '1STFLRSF', '2NDFLRSF',
    'YEARBUILT', 'YEARREMODADD',
    'GARAGECARS', 'GARAGEAREA',
    'FULLBATH', 'HALFBATH', 'BSMTFULLBATH',
    'FIREPLACES', 'POOLAREA',
    'OPENPORCHSF', 'SCREENPORCH', '3SSNPORCH',
]

df.drop(columns=cols_to_drop, inplace=True)

In [12]:
columns_qual_nom = [
    "MSSUBCLASS",
    "MSZONING",
    "STREET",
    "LANDCONTOUR",
    "UTILITIES",
    "LOTCONFIG",
    "NEIGHBORHOOD",
    "CONDITION1",
    "CONDITION2",
    "BLDGTYPE",
    "HOUSESTYLE",
    "ROOFSTYLE",
    "ROOFMATL",
    "EXTERIOR1ST",
    "EXTERIOR2ND",
    "FOUNDATION",
    "HEATING",
    "CENTRALAIR",
    "ELECTRICAL",
    "GARAGETYPE",
    "GARAGEFINISH",
    "PAVEDDRIVE",
    "SALETYPE",
    "SALECONDITION",
]

columns_qual_ord = [
    "LOTSHAPE",
    "OVERALLQUAL",
    "OVERALLCOND",
    "EXTERQUAL",
    "EXTERCOND",
    "BSMTQUAL",
    "BSMTCOND",
    "BSMTEXPOSURE",
    "BSMTFINTYPE1",
    "BSMTFINTYPE2",
    "HEATINGQC",
    "KITCHENQUAL",
    "FUNCTIONAL",
    "GARAGEQUAL",
    "GARAGECOND",
    "LANDSLOPE",
]

columns_quan_disc = [
    "YRSOLD",
    "MOSOLD",
    "BSMTHALFBATH",
    "BEDROOMABVGR",
    "KITCHENABVGR",
    "TOTRMSABVGRD",
    "GARAGEYRBLT",
    "HASBASEMENT",
    "HASGARAGE",
    "HASFIREPLACE",
    "HASPOOL",
]

columns_quan_cont = [
    "LOTFRONTAGE",
    "LOTAREA",
    "MASVNRAREA",
    "BSMTFINSF1",
    "BSMTFINSF2",
    "BSMTUNFSF",
    "LOWQUALFINSF",
    "GRLIVAREA",
    "WOODDECKSF",
    "ENCLOSEDPORCH",
    "MISCVAL",
    "TOTALLIVINGSF",
    "HOUSE_AGE",
    "REMODEL_AGE",
    "GARAGE_PROXY",
    "BATHROOM_INDEX",
    "TOTALPORCHSF",
    "SALEPRICE"
]

target = ["SALEPRICE"]


## Export for EDA

In [13]:
# EDA: null treatment only — no uppercase standardization, no encoding
df_eda = df_raw.copy()

null_pct_eda = df_eda.isnull().mean() * 100

high_missing_eda = null_pct_eda[null_pct_eda > 40].index
df_eda = df_eda.drop(columns=high_missing_eda)

df_eda['GarageYrBlt'] = df_eda['GarageYrBlt'].fillna(0)

low_missing_eda = null_pct_eda[(null_pct_eda > 0) & (null_pct_eda < 10)].index
df_eda = df_eda.dropna(subset=low_missing_eda)

medium_missing_eda = null_pct_eda[(null_pct_eda >= 10) & (null_pct_eda <= 40)].index
if len(medium_missing_eda) > 0:
    imputer_eda = KNNImputer(n_neighbors=5, weights='distance')
    df_eda[medium_missing_eda] = imputer_eda.fit_transform(df_eda[medium_missing_eda])



In [14]:
eda_dir = Path('../data/data_eda')
eda_dir.mkdir(parents=True, exist_ok=True)

for filename, out_name, is_train in [('train.csv', 'train_eda.csv', True), ('test.csv', 'test_eda.csv', False)]:
    raw = pd.read_csv(RAW_DATA_DIR / filename)
    d = raw.copy()

    null_pct = d.isnull().mean() * 100
    d = d.drop(columns=null_pct[null_pct > 40].index)
    d['GarageYrBlt'] = d['GarageYrBlt'].fillna(0)

    low_cols = null_pct[(null_pct > 0) & (null_pct < 10)].index
    if is_train:
        d = d.dropna(subset=low_cols)
    else:
        for col in low_cols:
            if col not in d.columns:
                continue
            if pd.api.types.is_numeric_dtype(d[col]):
                d[col] = d[col].fillna(d[col].median())
            else:
                d[col] = d[col].fillna(d[col].mode().iloc[0])

    med_cols = null_pct[(null_pct >= 10) & (null_pct <= 40)].index
    if len(med_cols) > 0:
        imp = KNNImputer(n_neighbors=5, weights='distance')
        d[med_cols] = imp.fit_transform(d[med_cols])

    d.to_csv(eda_dir / out_name, index=False)
    print(f'EDA saved ({out_name}) — shape: {d.shape}')

EDA saved (train_eda.csv) — shape: (1338, 75)
EDA saved (test_eda.csv) — shape: (1459, 74)
